# Ingest Observability Data into DOG with OpenTelemetry

**Module 2: Ingesting Observability Data** · [Quiz](quiz02_ingesting_observability_data.ipynb) · [Course README](../../README.md)

Run this notebook from JupyterLab launched at the course root. The `lab.shell(...)` cells use the course root to mount the YAML configuration files. The Doris FE log path is discovered from the running `doris-obs` container.

## 1. Send logs to DOG

Use the following command to write log data to DOG through OTel. It reads the actual FE log directory from the `doris-obs` container's host mount, so it does not depend on the current working directory or the location where the DOG stack was checked out. Each collector joins the course's `doris-obs` Docker network and reaches Doris by its container name, `doris-obs` (see `endpoint` and `mysql_endpoint` in the YAML files):

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


In [ ]:
lab.shell(r"""
set -euo pipefail

DORIS_CONTAINER="doris-obs"
CONFIG_DIR="level1/module02-data-ingest"
DORIS_FE_LOG_DIR="$(docker inspect --format '{{range .Mounts}}{{if eq .Destination "/opt/apache-doris/fe/log"}}{{.Source}}{{end}}{{end}}' "$DORIS_CONTAINER")"

if [ -z "$DORIS_FE_LOG_DIR" ]; then
  echo "Could not find the /opt/apache-doris/fe/log mount on the $DORIS_CONTAINER container."
  echo "Run module 1's prepare_environment() cell first, then rerun this cell."
  exit 1
fi

# Docker Desktop on macOS reports host mounts with a /host_mnt prefix.
DORIS_FE_LOG_DIR="${DORIS_FE_LOG_DIR#/host_mnt}"

DORIS_FE_LOG="$DORIS_FE_LOG_DIR/fe.log"
if [ ! -f "$DORIS_FE_LOG" ]; then
  echo "The mounted Doris FE log was not found at $DORIS_FE_LOG"
  exit 1
fi

docker rm -f doris-obs-log2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-log2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_logs.yaml:/etc/otel_for_logs.yaml:ro" \
  -v "$DORIS_FE_LOG:/var/log/fe.log:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_logs.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-log2doris
""", title="Start the Doris log collector")


**Expected result**

The command log ends with `collector=/doris-obs-log2doris status=running`. The collector reads `fe.log` from the beginning and then keeps following it. It does not remember how far it has read, so rerunning this cell loads the whole file again and duplicates the FE log rows in Doris.

### Check that the FE logs reached Doris

A running collector does not show that any rows reached Doris. The next cell looks for them in `otel_logs`: the filelog receiver records the file name in the `log.file.name` attribute, the `regex_parser` operator in `otel_for_logs.yaml` fills `severity_text` from each line, and the `resource` processor names the source of every row: `service_name` is `doris-fe`. The batch processor and the load into Doris take a few seconds, so the cell reruns the query until rows arrive.

In [ ]:
fe_logs = lab.wait_for_rows(
    """
    SELECT service_name, severity_text, COUNT(*) AS log_rows, MAX(timestamp) AS latest_log
    FROM otel_logs
    WHERE log_attributes['log.file.name'] = 'fe.log'
    GROUP BY service_name, severity_text
    ORDER BY service_name, severity_text
    """,
    title="FE log rows in otel_logs",
    hint="Check that doris-obs-log2doris is running with docker ps, and read its log with docker logs doris-obs-log2doris.",
)

**Expected result**

One row per severity in the FE log, all with `service_name = doris-fe`: usually `INFO` and `WARN`, plus `ERROR` when the FE has logged errors. The collector starts at the oldest line of `fe.log` and exports up to 10,000 records per batch (`send_batch_size` in `otel_for_logs.yaml`), so right after it starts, `latest_log` can lag behind the current time. Rerun the cell a little later: `log_rows` has grown, and `latest_log` is within a few seconds of the current time in UTC, because the FE keeps writing to `fe.log` and the collector keeps loading the new lines.

## 2. Send metrics to DOG

Use the following command to write metrics data to DOG through OTel. This example directly uses the metrics exposed by Doris:

In [ ]:
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-metric2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-metric2doris \
  --network doris-obs \
  -v "$PWD/$CONFIG_DIR/otel_for_metrics.yaml:/etc/otel_for_metrics.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_metrics.yaml

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-metric2doris
""", title="Start the Doris metric collector")


**Expected result**

The command log ends with `collector=/doris-obs-metric2doris status=running`.

### Check that the metrics reached Doris

The Prometheus receiver scrapes the two targets in `otel_for_metrics.yaml` every 15 seconds and names each series after the scrape job (`service_name = 'doris_cluster'`) and the target (`service_instance_id`). The Doris exporter writes one table per metric type and uses the configured name `otel_metrics` as the prefix: gauges go to `otel_metrics_gauge`, counters to `otel_metrics_sum`, and so on. The next cell waits until both targets have rows in `otel_metrics_gauge`.

In [ ]:
doris_metrics = lab.wait_for_rows(
    """
    SELECT service_instance_id AS scrape_target,
           COUNT(DISTINCT metric_name) AS metric_names,
           COUNT(*) AS data_points,
           MAX(timestamp) AS latest_point
    FROM otel_metrics_gauge
    WHERE service_name = 'doris_cluster'
    GROUP BY service_instance_id
    ORDER BY scrape_target
    """,
    title="Doris metrics in otel_metrics_gauge",
    hint="Check that doris-obs-metric2doris is running with docker ps, and read its log with docker logs doris-obs-metric2doris.",
    min_rows=2,
)

**Expected result**

Two rows, usually within 30 seconds: `doris-obs:8030` (the FE, about 70 metric names) and `doris-obs:8040` (the BE, over 100). `latest_point` is close to the current time in UTC, and `data_points` grows with every scrape when you rerun the cell.

## 3. Send traces to DOG

This collector receives spans over OTLP, adds the `deployment.environment.name` resource attribute (the `resource` processor in `otel_for_traces.yaml`), and writes the spans to `otel_traces`. Unlike the other two collectors, it publishes its OTLP/HTTP receiver on the host, at `127.0.0.1:24328` (or the port in `DORIS_OBS_TRACES_OTLP_HTTP_PORT`), so that the Python code in the next section can send spans to it. The DOG stack's own Collector listens on `127.0.0.1:24318`; spans sent there never pass through this collector or its configuration.

The first line of the cell checks that no other program uses the host port.

In [ ]:
lab.check_trace_collector_port()
lab.shell(r"""
set -euo pipefail

CONFIG_DIR="level1/module02-data-ingest"
docker rm -f doris-obs-traces2doris >/dev/null 2>&1 || true
docker run --rm -d \
  --name doris-obs-traces2doris \
  --network doris-obs \
  -p "127.0.0.1:${DORIS_OBS_TRACES_OTLP_HTTP_PORT:-24328}:4318" \
  -v "$PWD/$CONFIG_DIR/otel_for_traces.yaml:/etc/otel_for_traces.yaml:ro" \
  otel/opentelemetry-collector-contrib:0.144.0 \
  --config /etc/otel_for_traces.yaml

# The collector creates its tables in Doris before it opens the OTLP receiver.
# Wait until its log says that it is ready to receive data.
for attempt in $(seq 1 30); do
  docker logs doris-obs-traces2doris 2>&1 | grep "Everything is ready" >/dev/null && break
  sleep 1
done
docker logs doris-obs-traces2doris 2>&1 | grep -o "Everything is ready.*processing data."

docker inspect --format 'collector={{.Name}} status={{.State.Status}}' doris-obs-traces2doris
docker port doris-obs-traces2doris
""", title="Start the trace collector")


**Expected result**

The port check shows `24328` as `available`, or as `used by this course` when you rerun the cell. The collector creates its tables in Doris before it opens its receivers, so the cell waits for its log line `Everything is ready. Begin running and processing data.` The command log ends with that line, `collector=/doris-obs-traces2doris status=running`, and `4318/tcp -> 127.0.0.1:24328`.

## 4. Send a span and find it in Doris

The following code creates one span with the OpenTelemetry SDK, which the course's `requirements.txt` installs, and exports it to the trace collector over OTLP/HTTP.

In [ ]:
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

resource = Resource.create({"service.name": "python-demo-service"})

provider = TracerProvider(resource=resource)
# The trace collector's OTLP/HTTP receiver: http://127.0.0.1:24328 unless DORIS_OBS_TRACES_OTLP_HTTP_PORT is set.
processor = BatchSpanProcessor(OTLPSpanExporter(endpoint=f"{lab.trace_collector_url}/v1/traces"))
provider.add_span_processor(processor)
# Use this provider directly. The global trace.set_tracer_provider() accepts only one
# provider per Python process, so a rerun of this cell would keep using the first one.
tracer = provider.get_tracer("demo-tracer")

with tracer.start_as_current_span("demo-span") as span:
    span.set_attribute("custom.attribute", "hello-otel")
    span.set_attribute("http.status_code", 200)

# BatchSpanProcessor exports in the background. Export the queued span now.
provider.force_flush()
trace_id = format(span.get_span_context().trace_id, "032x")
print(f"Flushed the span. Trace ID: {trace_id}")


**Expected result**

The cell prints the 32-character trace ID of the span, for example `Flushed the span. Trace ID: 4bf92f3577b34da6a3ce929d0e0e4736`; yours differs on every run. `force_flush()` waits for the export attempt but does not report its outcome: when the collector is unreachable, the exporter only logs warnings such as `Failed to export spans batch` under the cell. Neither the print nor the flush shows that Doris stored the span, so the next cell looks for it.

### Find the span in Doris

Look the span up by its trace ID. The `environment` column reads the resource attribute that the trace collector adds. The Python code does not set it, so `course-lab` in that column shows that the span went through `doris-obs-traces2doris`.

In [ ]:
span_rows = lab.wait_for_rows(
    f"""
    SELECT trace_id, service_name, span_name, timestamp,
           resource_attributes['deployment.environment.name'] AS environment,
           span_attributes['custom.attribute'] AS custom_attribute
    FROM otel_traces
    WHERE trace_id = '{trace_id}'
    """,
    title="The span in otel_traces",
    hint="Check that doris-obs-traces2doris is running with docker ps, read its log with docker logs "
         "doris-obs-traces2doris, and look for export warnings under the previous cell.",
)

**Expected result**

One row, usually within 20 seconds: your trace ID, `python-demo-service`, `demo-span`, a `timestamp` in UTC, `environment = course-lab`, and `custom_attribute = hello-otel`. The delay comes from the collector's batch processor (`timeout: 10s` in `otel_for_traces.yaml`) and from Doris committing the load.

### Try it: change the collector configuration

The collector reads `otel_for_traces.yaml` only when it starts.

1. In `otel_for_traces.yaml`, change `value: course-lab` to another value, such as `value: my-laptop`, and save the file.
2. Rerun the cell in section 3 to start the collector again with the new file. If the cell fails, the collector could not start with the edited file: check the change, or undo it, and rerun the cell.
3. Rerun the two code cells in this section. The new span shows your value in `environment`.

## Next

Open [Quiz 2: Ingesting Observability Data](quiz02_ingesting_observability_data.ipynb), then continue to [Module 3: Visualizing Observability Data](../module03-data-visualization/lab03_data_visualization.ipynb).